In [1]:
import re
import warnings
import os
import pandas as pd
import pdfplumber

In [2]:
COLUMNS = [
    "intitule_emploi", "missions", "categorie_emploi", "type_emploi",
    "secteur_employeur", "activite_employeur", "lieu_emploi", "revenu",
    "autre_diplome",
]
# Colonnes du tableau, dans l'ordre du pdf. Il y en a 9 en 2017 : pas de colonne "Diplômé" (tous les répondants sont en formation initiale).
# On ajoutera donc une colonne "diplome" = "FI" à la fin pour garder le même schéma que 2024 et 2022.

PARAS_TABLEAUX = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines"
}
# Même réglage que pour les autres années : les séparations à prendre en compte sont les bordures du tableau.

TEXT_KW = {
    "x_tolerance": 1.5,
    "y_tolerance": 2
}
# Les espaces entre les mots sont très serrés (environ 2 points).
# Une faible tolérance évite de coller plusieurs mots ensemble.

# Bande verticale contenant le pied de page (« Mention … - Parcours … » puis « n° retour au sommaire »).
# La largeur est celle de la page entière. Seule la limite haute est fixée.
FOOTER_Y_MIN = 550

**Comment ces paramètres ont été choisis.** Comme pour 2016 et 2022, l'en-tête et l'encadré de stats ne sont pas lus dans une zone fixe : on lit le texte de la page entière et on s'ancre sur des mots-clés (`RÉPERTOIRE DES EMPLOIS DES DIPLÔMÉS DE MASTER`, `Mention`, `Parcours`, `Promotion`, `Situation au`...).
La seule zone fixée est le pied de page (`FOOTER_Y_MIN`) : il est présent sur **toutes** les pages, y compris les pages « suite », et donne à la fois la mention et le parcours. Il permet de rattacher chaque page à sa formation et de contrôler l'en-tête.
Les particularités du pdf 2017 (formulations, pages sans en-tête, taux décimaux, types d'emploi `NR`) ont été repérées en inspectant le fichier (positions des mots, tableaux détectés) avec l'aide de Claude, puis validées sur l'ensemble des pages.

In [3]:
HEADER_RE = re.compile(
    r"RÉPERTOIRE DES EMPLOIS DES DIPLÔMÉS DE MASTER\s*"
    r"Mention\s+(?P<mention>.+?)\s*"
    r"Parcours\s+(?P<parcours>.+?)\s*"
    r"Promotion\s+(?P<promotion>\d{4})\s+en\s+(?P<population>[^\n]+?)\s*\n"
    r"\s*Situation au\s+(?P<situation>[^\n]+)",
    re.S,
)
# Exemple de ce qu'on cherche :
#   Mention Arts / Parcours Cinéma et nouvelles images /
#   Promotion 2017 en formation initiale / Situation au 1er décembre 2019

# Pied de page : « Mention <mention> - Parcours <parcours> » puis, à la ligne, « <n° de page> retour au sommaire ».
FOOTER_RE = re.compile(
    r"Mention\s+(?P<mention>.+?)\s+-\s+Parcours\s+(?P<parcours>.+?)\s*\n\s*\d+\s+retour au sommaire",
    re.S,
)

# Un motif de recherche est associé à chaque statistique.
STATS_RE = {
    "n_concernes": r"concern\S*\s+par l['’]enquête\s*:\s*(\d+)",
    "taux_reponse": r"Taux de réponse\s*:\s*(\d+(?:[.,]\d+)?)\s*%",   # le taux peut être décimal (33,3%)
    "n_en_emploi": r"En emploi\s+(\d+)",
    "n_recherche_emploi": r"En recherche d['’]emploi\s+(\d+)",
    "n_en_etudes": r"En études\s+(\d+)",
    "n_autre_situation": r"Autre situation\s+(\d+)",
    "n_total_repondants": r"Total\s+(\d+)",
    "n_emploi_stable": r"Emploi stable\s*:?\s*(\d+)",          # ":" facultatif (absent en 2017)
    "n_emploi_non_stable": r"Emploi non stable\s*:?\s*(\d+)",
}

# Pour ces catégories, une ligne absente de l'encadré signifie zéro.
# C'est vérifié sur ce pdf : la somme des situations = total des répondants pour toutes les formations.
BOX_COUNTS = [
    "n_en_emploi",
    "n_recherche_emploi",
    "n_en_etudes",
    "n_autre_situation",
    "n_emploi_stable",
    "n_emploi_non_stable",
]

In [4]:
def clean(text):
    """Nettoie le texte d'une cellule et remplace certaines valeurs par None.
    Le but est de garantir que les données récupérées sont propres et homogènes avant d'être stockées."""
    if text is None:
        return None

    # Remplacer les suites d'espaces et de retours à la ligne par un espace.
    text = re.sub(r"\s+", " ", text).strip()

    # Considérer quelques valeurs comme des données manquantes :
    if text in ("", "-", "NR"):
        return None

    return text


def cle_formation(infos):
    """Clé (mention, parcours) normalisée, pour comparer l'en-tête et le pied de page
    (les apostrophes typographiques et les espaces peuvent différer)."""
    def norm(s):
        return re.sub(r"\s+", " ", s.replace("’", "'")).strip()
    return (norm(infos["mention"]), norm(infos["parcours"]))

In [5]:
def parse_header(page):
    """Renvoie les informations de l'en-tête, ou None si la page n'a pas d'en-tête de formation.
    En 2017, seule la première page d'une formation en a un : les pages « suite » n'en ont pas."""
    text = page.extract_text(**TEXT_KW) or ""  # texte de toute la page
    match = HEADER_RE.search(text)

    if not match:
        return None

    return {
        key: re.sub(r"\s+", " ", value).strip()
        for key, value in match.groupdict().items()
    }

In [6]:
def parse_footer(page):
    """Renvoie la mention et le parcours écrits en pied de page, ou None.
    Le pied de page existe sur toutes les pages : il permet d'identifier les pages « suite »."""
    text = page.crop(
        (0, FOOTER_Y_MIN, page.width, page.height)
    ).extract_text(**TEXT_KW) or ""

    match = FOOTER_RE.search(text)

    if not match:
        return None

    return {
        key: re.sub(r"\s+", " ", value).strip()
        for key, value in match.groupdict().items()
    }

In [7]:
def parse_stats(page):  # Même principe qu'en 2016 et 2022 : texte de la page entière
    """Lit les statistiques présentes dans l'encadré de la première page d'une formation."""
    text = page.extract_text(**TEXT_KW) or ""

    stats = {}

    # Utilisation des expressions régulières pour chercher les stats et les stocker dans le dictionnaire :
    for key, pattern in STATS_RE.items():
        match = re.search(pattern, text)

        if match:
            valeur = match.group(1).replace(",", ".")   # virgule décimale -> point
            stats[key] = float(valeur) if key == "taux_reponse" else int(valeur)
        else:
            stats[key] = None

    # Une ligne absente signifie zéro pour ces catégories,
    # si le total des répondants est présent.
    if stats["n_total_repondants"] is not None:
        for key in BOX_COUNTS:
            if stats[key] is None:
                stats[key] = 0

    # Idem si le PDF signale un effectif insuffisant : certaines formations n'ont pas de tableau
    # (pas assez de répondants) et c'est cette mention qui les caractérise.
    stats["effectif_insuffisant"] = "Effectif insuffisant" in text

    return stats

In [8]:
def extract_jobs(page):
    """Extrait les lignes du tableau des emplois d'une page."""
    rows = []

    # Rechercher les tableaux présents sur la page avec les paramètres spécifiés à pdfplumber
    for table in page.find_tables(PARAS_TABLEAUX):

        # Ignorer les tableaux qui n'ont pas le nombre attendu de colonnes :
        # cela permet notamment d'ignorer le petit tableau de statistiques (2 colonnes)
        if len(table.rows[0].cells) != len(COLUMNS):
            continue

        for row in table.rows:
            values = []

            # Lire et nettoyer chaque cellule de la ligne.
            for cell in row.cells:
                if cell:
                    value = clean(
                        page.crop(cell).extract_text(**TEXT_KW)
                    )
                else:
                    value = None

                values.append(value)

            # Ne pas conserver la ligne qui contient les titres des colonnes
            # (elle est répétée en haut de chaque page, y compris les pages « suite ») :
            if values[0] is not None and values[0].startswith("Intitulé"):
                continue

            # Enfin on lie les valeurs aux bonnes colonnes :
            rows.append(dict(zip(COLUMNS, values)))

    return rows

In [9]:
def extract(pdf_path, first=1, last=None):
    """Pipeline complet pour le répertoire 2017."""
    jobs = []
    formations = []
    pages_ignorees = []

    current_header = None
    current_stats = {}

    with pdfplumber.open(pdf_path) as pdf:

        # Si la dernière page n'est pas précisée, lire jusqu'à la fin du PDF.
        if last is None:
            last = len(pdf.pages)

        for page_no in range(first, last + 1):
            page = pdf.pages[page_no - 1]

            header = parse_header(page)
            footer = parse_footer(page)
            page_jobs = extract_jobs(page)

            if header is not None:
                # Page avec en-tête : nouvelle formation, on lit ses statistiques.
                stats = parse_stats(page)

                # Sécurité : l'en-tête et le pied de page doivent désigner la même formation.
                if footer is None or cle_formation(footer) != cle_formation(header):
                    warnings.warn(
                        f"Page {page_no} : en-tête et pied de page différents ({header['parcours']})."
                    )

                current_header = header
                current_stats = stats

                formations.append({
                    "page": page_no,
                    **header,
                    **stats,
                    "n_pages": 1,
                    "n_lignes_extraites": 0
                })

            else:
                # Pas d'en-tête : c'est soit une page « suite » (elle contient un tableau),
                # soit une page sans intérêt pour nous (couverture, présentation, sommaire, page de fin...).
                if not page_jobs or current_header is None:
                    pages_ignorees.append(page_no)
                    continue

                # Sécurité : le pied de page doit rappeler la formation en cours.
                if footer is None or cle_formation(footer) != cle_formation(current_header):
                    warnings.warn(
                        f"Page {page_no} : le pied de page ne correspond pas à la formation en cours "
                        f"({current_header['parcours']}). Rattachement à vérifier."
                    )

                # Une page « suite » reprend l'en-tête et les statistiques de la formation en cours.
                header, stats = current_header, current_stats
                formations[-1]["n_pages"] += 1

            formations[-1]["n_lignes_extraites"] += len(page_jobs)

            for job in page_jobs:
                jobs.append({
                    "page": page_no,
                    **header,
                    **stats,
                    **job
                })

    # Garde-fou : ne pas échouer en silence si le format du pdf n'est pas celui attendu.
    if not formations:
        warnings.warn(
            "Aucune formation reconnue : le format du pdf ne correspond sans doute pas à celui attendu "
            "(vérifier HEADER_RE)."
        )
    print(f"Pages ignorées ({len(pages_ignorees)}) : {pages_ignorees}")

    # Transforme nos listes en 2 df : df est notre base globale, df_formations contient les infos propres à chaque formation
    df_formations = pd.DataFrame(formations)
    df = pd.DataFrame(jobs)

    if not df_formations.empty:
        # Contrôle : le nombre de lignes extraites doit être égal à l'effectif en emploi
        # (sauf pour les formations à « effectif insuffisant », qui n'ont pas de tableau).
        df_formations["ecart_lignes"] = (
            df_formations["n_lignes_extraites"] - df_formations["n_en_emploi"]
        )
        df_formations.loc[df_formations["effectif_insuffisant"], "ecart_lignes"] = None

    if not df.empty:
        raw = df["revenu"].fillna("")

        # Indiquer si la valeur du revenu mentionne un temps partiel.
        df["temps_partiel"] = raw.str.contains(
            "Temps partiel",
            case=False
        )

        # Convertir le revenu en nombre. Les valeurs non numériques deviennent NaN.
        df["revenu_net"] = pd.to_numeric(
            raw.str.replace(r"\s+", "", regex=True),
            errors="coerce"
        )

        # Transformer le type d'emploi en variable booléenne (reste vide si le type est « NR »).
        df["emploi_stable"] = df["type_emploi"].map({
            "Stable": True,
            "Non stable": False
        })

        # Le pdf 2017 ne concerne que la formation initiale : on crée la colonne "diplome"
        # pour garder le même schéma que les notebooks 2024 et 2022.
        df["diplome"] = "FI"

        # Supprimer la colonne brute.
        df = df.drop(columns="revenu")

    return df, df_formations

In [11]:
FIRST_PAGE= 1
LAST_PAGE = None
dossier = r"C:/Users/33698/Desktop/Cours/Master/M2 DS SIAD/ML/projet/data/donnée brute/2018"
dossier_sortie = r"C:/Users/33698/Desktop/Cours/Master/M2 DS SIAD/ML/projet/data/extract"


os.makedirs(dossier_sortie, exist_ok=True)
for fichier in os.scandir(dossier):

    if fichier.is_file() and fichier.name.lower().endswith(".pdf"):
        chemin_pdf = fichier.path
        nom = os.path.splitext(fichier.name)[0]
        df, df_formations = extract(chemin_pdf,FIRST_PAGE,LAST_PAGE)
        df.to_excel(os.path.join(dossier_sortie, f"{nom}_extract.xlsx"),index=False)
        df_formations.to_excel(os.path.join(dossier_sortie, f"{nom}_formations.xlsx"),index=False)
        print(df.shape)
print("fin du traitement")

C:\Users\33698\AppData\Local\Temp\ipykernel_16352\231264859.py:74: UserWarning: Aucune formation reconnue : le format du pdf ne correspond sans doute pas à celui attendu (vérifier HEADER_RE).
  warnings.warn(


Pages ignorées (147) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147]
(0, 0)
Pages ignorées (176) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82,